In [ ]:
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

from openai import OpenAI

client = OpenAI(api_key="")

c:\Users\vinee\Desktop\AI research Engineer SHL\3MarEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("processed_assessments_clean.json","r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Assessments:",len(df))
df.head()

Assessments: 377


,name,url,assessed_skills_norm,cognitive_dimensions_norm,target_roles_norm,industry_tags,final_duration,experience_numeric,embedding
0,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,"[portability, application development, perform...",[],"[programmers, application developers]",[Information Technology],30.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.019559728, -0.0246569775, 0.0208423324, -0..."
1,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,[],[],[],[],NaN,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[0.0381163061, 0.0718954131, 0.0741915777, 0.0..."
2,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,"[programming knowledge, validation, routing, s...",[],"[full stack developer, software engineer, tech...",[Information Technology],17.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.0091075459, -0.0168671068, 0.0248628519, -..."
3,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,"[wpf data bindings, mvvm pattern, events, data...",[],"[full stack developer, net developer, software...",[Information Technology],5.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.0426559374, 0.017746916, 0.0271624029, -0...."
4,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,"[behaviors, messages, clients, services and co...",[],"[full stack developer, net developer - wcf, so...",[Information Technology],11.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.0292214733, -0.0284445733, 0.0452607013, -..."


In [3]:
def build_structured_text(row):

    parts = []

    if isinstance(row.get("assessed_skills_norm"), list):
        parts.append("Skills: " + ", ".join(row["assessed_skills_norm"]))

    if isinstance(row.get("target_roles_norm"), list):
        parts.append("Roles: " + ", ".join(row["target_roles_norm"]))

    if isinstance(row.get("cognitive_dimensions_norm"), list):
        parts.append("Cognitive: " + ", ".join(row["cognitive_dimensions_norm"]))

    if isinstance(row.get("industry_tags"), list):
        parts.append("Industry: " + ", ".join(row["industry_tags"]))

    if row.get("description"):
        parts.append("Description: " + row["description"])

    return " ".join(parts)

df["retrieval_text"] = df.apply(build_structured_text,axis=1)

In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

c:\Users\vinee\Desktop\AI research Engineer SHL\3MarEnv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
embeddings = model.encode(
    df["retrieval_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 12/12 [00:05<00:00,  2.17it/s]


In [6]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_features=50000
)

tfidf_matrix = tfidf.fit_transform(df["retrieval_text"])
tfidf_matrix = normalize(tfidf_matrix)

In [7]:
def reason_about_query(query):

    prompt = f"""
You are an expert HR assessment consultant.

Your job is to understand a hiring query or job description
and determine what assessments should evaluate.

Think like a recruiter.

Extract the following information.

Return JSON with fields:

role:
The job role being hired for.

seniority:
entry, mid, senior, executive if possible.

competencies:
Key competencies that should be assessed.

technical_skills:
Technical tools or technologies mentioned.

assessment_types:
Types of assessments appropriate for this role.
Examples: personality, cognitive ability, leadership,
technical knowledge, communication, situational judgement.

max_duration:
Maximum assessment duration if mentioned.

Query:
{query}

Return JSON only.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    text = response.choices[0].message.content

    m = re.search(r"\{.*\}",text,re.DOTALL)

    if m:
        return json.loads(m.group())

    return {}

In [8]:
def build_query_text(parsed):

    parts = []

    parts.append(parsed.get("role",""))

    parts += parsed.get("competencies",[])
    parts += parsed.get("technical_skills",[])
    parts += parsed.get("assessment_types",[])

    return " ".join(parts)

In [9]:
def retrieve_candidates(query_text,top_k=100):

    q_emb = model.encode(query_text,normalize_embeddings=True)

    sims = cosine_similarity([q_emb],embeddings)[0]

    idx = np.argsort(sims)[::-1][:top_k]

    return idx,sims

In [10]:
def hybrid_score(query_text,idx,sims):

    q_tfidf = tfidf.transform([query_text])

    tfidf_scores = (tfidf_matrix[idx] @ q_tfidf.T).toarray().ravel()

    scores = 0.7*sims[idx] + 0.3*tfidf_scores

    return scores

In [20]:
def ensure_coverage(results, competencies):

    if not competencies:
        return results.head(10)

    selected_indices = []
    covered = set()

    for idx, row in results.iterrows():

        text = str(row["retrieval_text"]).lower()

        comp_matches = [c for c in competencies if c.lower() in text]

        if comp_matches:
            selected_indices.append(idx)
            covered.update(comp_matches)

        if len(selected_indices) >= 10:
            break

    # Fill remaining slots with top ranked results
    if len(selected_indices) < 10:

        for idx in results.index:
            if idx not in selected_indices:
                selected_indices.append(idx)

            if len(selected_indices) >= 10:
                break

    return results.loc[selected_indices].head(10)

In [12]:
def recommend(query):

    parsed = reason_about_query(query)

    query_text = build_query_text(parsed)

    idx,sims = retrieve_candidates(query_text)

    scores = hybrid_score(query_text,idx,sims)

    res = df.iloc[idx].copy()
    res["score"] = scores

    res = res.sort_values("score",ascending=False)

    res = ensure_coverage(res,parsed.get("competencies",[]))

    return res

In [36]:
query = """Content Writer required, expert in English and SEO."""

print(reason_about_query(query))

{'role': 'Content Writer', 'seniority': 'mid', 'competencies': ['writing skills', 'SEO knowledge', 'creativity', 'attention to detail', 'time management'], 'technical_skills': ['SEO tools', 'content management systems', 'keyword research tools'], 'assessment_types': ['writing assessment', 'technical knowledge', 'creativity', 'communication'], 'max_duration': None}


In [14]:
def slug(url):

    if not isinstance(url,str):
        return ""

    url = url.lower()

    m = re.search(r"/view/([^/]+)",url)

    if m:
        return m.group(1)

    return url

In [16]:
def recall_at_10(pred,true):

    p = {slug(u) for u in pred}
    t = {slug(u) for u in true}

    if not t:
        return 0

    return len(p & t)/len(t)

In [17]:
eval_df = pd.read_excel("Gen_AI Dataset.xlsx",sheet_name="Train-Set")

grouped = eval_df.groupby("Query")["Assessment_url"].apply(list).to_dict()

In [18]:
def evaluate():

    recalls=[]

    for q,urls in tqdm(grouped.items()):

        res = recommend(q)

        preds = res["url"].tolist()

        r = recall_at_10(preds,urls)

        recalls.append(r)

    return np.mean(recalls)

In [21]:
mean_recall = evaluate()

print("Mean Recall@10:",mean_recall)

100%|██████████| 10/10 [00:29<00:00,  2.91s/it]

Mean Recall@10: 0.2166666666666667


In [22]:
rows=[]

for q in grouped:

    res = recommend(q)

    for rank,row in enumerate(res.itertuples(),1):

        rows.append({
            "query":q,
            "rank":rank,
            "name":row.name,
            "url":row.url,
            "score":row.score
        })

pd.DataFrame(rows).to_excel("recommendations.xlsx",index=False)

In [23]:
rows = []

for query, true_urls in grouped.items():

    # ground truth slugs
    true_slugs = {slug(u) for u in true_urls}

    # model recommendations
    res = recommend(query)

    for rank, row in enumerate(res.itertuples(), start=1):

        pred_slug = slug(row.url)

        rows.append({
            "query": query,
            "rank": rank,
            "assessment_name": row.name,
            "predicted_url": row.url,
            "score": row.score,
            "predicted_slug": pred_slug,
            "is_relevant": pred_slug in true_slugs,
            "ground_truth_urls": ", ".join(true_urls)
        })

df_compare = pd.DataFrame(rows)

df_compare.head()

,query,rank,assessment_name,predicted_url,score,predicted_slug,is_relevant,ground_truth_urls
0,Based on the JD below recommend me assessment ...,1,Managerial Scenarios Candidate Report,https://www.shl.com/products/product-catalog/v...,0.421509,managerial-scenarios-candidate-report,False,https://www.shl.com/solutions/products/product...
1,Based on the JD below recommend me assessment ...,2,Managerial Scenarios Narrative Report,https://www.shl.com/products/product-catalog/v...,0.421509,managerial-scenarios-narrative-report,False,https://www.shl.com/solutions/products/product...
2,Based on the JD below recommend me assessment ...,3,Managerial Scenarios Profile Report,https://www.shl.com/products/product-catalog/v...,0.421509,managerial-scenarios-profile-report,False,https://www.shl.com/solutions/products/product...
3,Based on the JD below recommend me assessment ...,4,SAP Business Objects WebI (New),https://www.shl.com/products/product-catalog/v...,0.405592,sap-business-objects-webi-new,False,https://www.shl.com/solutions/products/product...
4,Based on the JD below recommend me assessment ...,5,Statistical Analysis System (New),https://www.shl.com/products/product-catalog/v...,0.397930,statistical-analysis-system-new,False,https://www.shl.com/solutions/products/product...


In [24]:
df_compare.to_excel("recommendation_comparison.xlsx", index=False)

print("Saved: recommendation_comparison.xlsx")

Saved: recommendation_comparison.xlsx
